# JSON

### JSON -> dict

In [1]:
# A partir d'un JSON sous format texte (ou JSON string) : 
obj = """ 
{"name": "Wes",
 "places_lived": ["United States", "Spain", "Germany"], 
 "pet": null,
 "siblings": [{"name": "Scott", "age": 25, "pet": "Zuko"},
              {"name": "Katie", "age": 33, "pet": "Cisco"}]
}
"""

In [ ]:
# Créer un dict à partir d'une JSON string
import json

result = json.loads(obj)
result

{'name': 'Wes',
 'places_lived': ['United States', 'Spain', 'Germany'],
 'pet': None,
 'siblings': [{'name': 'Scott', 'age': 25, 'pet': 'Zuko'},
  {'name': 'Katie', 'age': 33, 'pet': 'Cisco'}]}

In [ ]:
result["name"]

'Wes'

In [ ]:
# Créer un dict à partir d'un fichier JSON
result_open = json.load(open("../data/json_ex.json"))
result_open

{'name': 'Wes',
 'places_lived': ['United States', 'Spain', 'Germany'],
 'pet': None,
 'siblings': [{'name': 'Scott', 'age': 25, 'pet': 'Zuko'},
  {'name': 'Katie', 'age': 33, 'pet': 'Cisco'}]}

### dict -> JSON

In [21]:
json_string = json.dumps(result, indent=4)
print(json_string)

{
    "name": "Wes",
    "places_lived": [
        "United States",
        "Spain",
        "Germany"
    ],
    "pet": null,
    "siblings": [
        {
            "name": "Scott",
            "age": 25,
            "pet": "Zuko"
        },
        {
            "name": "Katie",
            "age": 33,
            "pet": "Cisco"
        }
    ]
}


### DF -> JSON

In [2]:
from json import loads, dumps
import pandas
df = pandas.DataFrame(
    [["a", "b"], ["c", "d"]],
    index=["row 1", "row 2"],
    columns=["col 1", "col 2"],
)
df

,col 1,col 2
row 1,a,b
row 2,c,d


In [27]:
print(df.to_json(indent=4, orient='columns'))

{
    "col 1":{
        "row 1":"a",
        "row 2":"c"
    },
    "col 2":{
        "row 1":"b",
        "row 2":"d"
    }
}


In [28]:
print(df.to_json(indent=4, orient='split'))

{
    "columns":[
        "col 1",
        "col 2"
    ],
    "index":[
        "row 1",
        "row 2"
    ],
    "data":[
        [
            "a",
            "b"
        ],
        [
            "c",
            "d"
        ]
    ]
}


In [29]:
print(df.to_json(indent=4, orient='records'))

[
    {
        "col 1":"a",
        "col 2":"b"
    },
    {
        "col 1":"c",
        "col 2":"d"
    }
]


In [30]:
print(df.to_json(indent=4, orient='values'))

[
    [
        "a",
        "b"
    ],
    [
        "c",
        "d"
    ]
]


In [31]:
print(df.to_json(indent=4, orient='table'))

{
    "schema":{
        "fields":[
            {
                "name":"index",
                "type":"string"
            },
            {
                "name":"col 1",
                "type":"string"
            },
            {
                "name":"col 2",
                "type":"string"
            }
        ],
        "primaryKey":[
            "index"
        ],
        "pandas_version":"1.4.0"
    },
    "data":[
        {
            "index":"row 1",
            "col 1":"a",
            "col 2":"b"
        },
        {
            "index":"row 2",
            "col 1":"c",
            "col 2":"d"
        }
    ]
}


### JSON -> DF et dict -> DF

### read_json

In [11]:
pandas.read_json("../data/read_example.json")

,id,name,age,city
0,1,Alice,30,Paris
1,2,Bob,25,Lyon
2,3,Charlie,35,Marseille


In [ ]:
# Limites de read_json
pandas.read_json("../data/normalize_example.json")

,id,name,location,hobbies
0,1,Alice,"{'city': 'Paris', 'postal_code': '75000'}","[reading, cycling]"
1,2,Bob,"{'city': 'Lyon', 'postal_code': '69000'}","[chess, running]"


### json_normalize

In [24]:
import json 

with open("../data/normalize_example.json") as f:
    data = json.load(f)

print(json.dumps(data, indent=4))

pandas.json_normalize(data)

[
    {
        "id": 1,
        "name": "Alice",
        "location": {
            "city": "Paris",
            "postal_code": "75000"
        },
        "hobbies": [
            "reading",
            "cycling"
        ]
    },
    {
        "id": 2,
        "name": "Bob",
        "location": {
            "city": "Lyon",
            "postal_code": "69000"
        },
        "hobbies": [
            "chess",
            "running"
        ]
    }
]


,id,name,hobbies,location.city,location.postal_code
0,1,Alice,"[reading, cycling]",Paris,75000
1,2,Bob,"[chess, running]",Lyon,69000


In [ ]:
# Cas plus complexe
# On souhaite une ligne par commande ("orders")
json_data = [
    {
        "user": "Alice",
        "orders": [
            {"id": 1, "item": "Coffee"},
            {"id": 2, "item": "Tea"}
        ]
    },
    {
        "user": "Bob",
        "orders": [
            {"id": 3, "item": "Juice"}
        ]
    }
]
# il faut ajouter des paramètres à la fonction json_normalize
df = pandas.json_normalize(json_data)#, record_path='orders', meta=['user'])
df

,user,orders
0,Alice,"[{'id': 1, 'item': 'Coffee'}, {'id': 2, 'item'..."
1,Bob,"[{'id': 3, 'item': 'Juice'}]"


In [ ]:
# Ici on veut une ligne par commande ("orders") et on veut garder la colonne "user"

# On peut utiliser le paramètre "record_path" pour indiquer la colonne à "normaliser"
# et le paramètre "meta" pour indiquer les colonnes à garder

df = pandas.json_normalize(json_data, record_path='orders', meta=['user'])
df

,id,item,user
0,1,Coffee,Alice
1,2,Tea,Alice
2,3,Juice,Bob


In [ ]:
# Si on veut garder toutes les colonnes de la table d'origine, on peut utiliser le paramètre "meta_prefix"
data = [
    {
        "id": 1,
        "name": "Cole Volk",
        "fitness": {"height": 130, "weight": 60},
    },
    {"name": "Mark Reg", "fitness": {"height": 130, "weight": 60}},
    {
        "id": 2,
        "name": "Faye Raker",
        "fitness": {"height": 130, "weight": 60},
    },
]


# rappel : le construction DataFrame présente des limites

pandas.DataFrame(data)

,id,name,fitness
0,1.0,Cole Volk,"{'height': 130, 'weight': 60}"
1,NaN,Mark Reg,"{'height': 130, 'weight': 60}"
2,2.0,Faye Raker,"{'height': 130, 'weight': 60}"


In [ ]:
# On utilise le paramètre "max_level" pour indiquer le niveau de profondeur à normaliser
# pandas.json_normalize(data, max_level=0) donne un résultat identique à la cellule ci-dessus
pandas.json_normalize(data, max_level=1)

,id,name,fitness.height,fitness.weight
0,1.0,Cole Volk,130,60
1,NaN,Mark Reg,130,60
2,2.0,Faye Raker,130,60


In [ ]:
data_nested = [
    {
        "id": 1,
        "name": "Cole Volk",
        "fitness": {"height": 130, "weight": 60}, "activities": ["tennis", "basketball"]
    },
    {"name": "Mark Reg", "fitness": {"height": 130, "weight": 60}, "activities": ["tennis", "football"]},
    {
        "id": 2,
        "name": "Faye Raker",
        "fitness": {"height": 130, "weight": 60}, "activities": ["tennis"]
    },
]
# Nouvelle version du JSON
# On observe maintenant une liste ("activities") imbriquée dans le JSON 
# On cherche également à "éclater" cette liste, pour avoir une ligne par activité

,id,name,activities,fitness.height,fitness.weight
0,1.0,Cole Volk,"[tennis, basketball]",130,60
1,NaN,Mark Reg,"[tennis, football]",130,60
2,2.0,Faye Raker,[tennis],130,60


In [9]:
df.explode("activities")

,id,name,activities,fitness.height,fitness.weight
0,1.0,Cole Volk,tennis,130,60
0,1.0,Cole Volk,basketball,130,60
1,NaN,Mark Reg,tennis,130,60
1,NaN,Mark Reg,football,130,60
2,2.0,Faye Raker,tennis,130,60


In [31]:
# Retour sur les parametres record_path et meta

data = [
    {
        "company_id": 1,
        "company_name": "TechCorp",
        "employees": [
            {
                "emp_id": 101,
                "name": "Alice",
                "projects": [
                    {"name": "Project A", "duration": 6},
                    {"name": "Project B", "duration": 3}
                ]
            },
            {
                "emp_id": 102,
                "name": "Bob",
                "projects": [
                    {"name": "Project C", "duration": 12}
                ]
            }
        ]
    },
    {
        "company_id": 2,
        "company_name": "BizDev",
        "employees": [
            {
                "emp_id": 201,
                "name": "Charlie",
                "projects": []
            }
        ]
    }
]


# On souhaite une ligne par projet ("projects") et on veut garder les infos du salarié et de l'entreprise
 
df = pandas.json_normalize(
    data,
    record_path=["employees", "projects"],
    meta=["company_id", "company_name", ["employees", "name"], ["employees", "emp_id"]],
)
df

,name,duration,company_id,company_name,employees.name,employees.emp_id
0,Project A,6,1,TechCorp,Alice,101
1,Project B,3,1,TechCorp,Alice,101
2,Project C,12,1,TechCorp,Bob,102
